In [ ]:
# ===== 0. Cấu hình
import os
BRANCH = "htstep_lora"                  # nhánh trên GitHub (yêu cầu ghi htstep-lora; tên thật có gạch dưới)
REPO_URL = "https://github.com/im-xiaoming/Omni-Retriever-Custom.git"
REPO = "/content/Omni-fix"
HF_CLIPS = ""                           # repo HF dataset RIÊNG của nhóm cho clip, private, ví dụ "<user>/htstep-omni-clips"
HTSTEP_COMMIT = "cca1dea"               # github.com/facebookresearch/htstep: annotations.json + taxonomy.csv gốc
HT = "/content/htstep"
WAVE_HOME = "/content/WAVE_HOME"
assert HF_CLIPS and "/" in HF_CLIPS, "điền HF_CLIPS (repo HF dataset riêng của nhóm) rồi chạy lại ô này"
WORK = "/content/precut"                # mp4 tạm, clip, shard trước khi upload (đĩa local)
JOBS = 0                                # 0 = min(nproc, RAM GB / 2.5)
DRIVE_LOG = "/content/drive/MyDrive/omniretriever/htstep_lora/precut"   # log + file kiểm tra trên Drive
YC2_IDS = "6MBctYaMU8U"                 # video HT-Step trùng YouCook2 (Colab không có file YC2)

In [ ]:
# ===== 1. Drive, code, thư viện (requirements-htstep.txt), máy
import os, sys, json, glob, re, shutil, signal, subprocess, time, hashlib
from collections import Counter
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")     # Colab Secrets: HF_TOKEN (quyền ghi repo HF_CLIPS)


def sh(cmd, log=None, cwd=None):
    """Chạy lệnh shell, in output (và ghi thêm vào log); nút Stop gửi Ctrl-C cho lệnh để nó dừng gọn."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    f = open(log, "a", encoding="utf-8") if log else None
    try:
        try:
            for line in p.stdout:
                print(line, end="", flush=True)
                if f:
                    f.write(line)
                    f.flush()
        except KeyboardInterrupt:
            os.killpg(p.pid, signal.SIGINT)
            for line in p.stdout:
                print(line, end="", flush=True)
                if f:
                    f.write(line)
            p.wait()
            raise
        return p.wait()
    finally:
        if f:
            f.close()


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()


if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
sh(f"git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only && git log -1 --format='commit %H %s'")
assert sh("pip install -q -r requirements-htstep.txt") == 0
sh("""python -c "import sys, platform, decord, av, transformers, peft, huggingface_hub, torch; print('python', platform.python_version(), '| decord', decord.__version__, '| av', av.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| hub', huggingface_hub.__version__, '| torch', torch.__version__)" """)
sh("ffmpeg -version | head -1")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2 ** 30
print(f"nproc {os.cpu_count()} | RAM {ram:.1f} GB | đĩa trống /content {shutil.disk_usage('/content').free / 2 ** 30:.0f} GB")
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", GPU or "không có")
if GPU:
    print("Lưu ý: notebook này chỉ cần runtime CPU (High-RAM nếu có)")
os.makedirs(DRIVE_LOG, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

In [ ]:
# ===== 2. Sinh split từ nhãn gốc (convert_htstep.py split), kiểm items_*.jsonl trùng bản đã commit
if not os.path.isdir(HT):
    sh(f"git clone -q https://github.com/facebookresearch/htstep.git {HT}")
sh(f"git checkout -q {HTSTEP_COMMIT} && git log -1 --format='htstep %H'", cwd=HT)
# --no-verify: tar_index_tars.json (kích thước tar) chỉ có ở máy local; range read của precut vẫn kiểm 206 + đủ byte
rc = sh(f"python scripts/convert_htstep.py split --htstep-dir {HT}/data --tar-index data/htstep/tar_index.csv "
        f"--no-verify --yc2 {YC2_IDS} --uniav-feat-index '' --out-dir data/htstep", log=f"{DRIVE_LOG}/split.log")
assert rc == 0, "split lỗi: gửi split.log cho agent"
same = subprocess.run(["git", "-C", REPO, "diff", "--quiet", "--", "data/htstep/items_train.jsonl",
                       "data/htstep/items_val.jsonl"]).returncode == 0
print("CHECK items_unchanged", "OK" if same else "FAIL")
assert same, "items_*.jsonl khác bản đã commit: dừng, gửi split.log cho agent"
S = json.load(open(f"{REPO}/data/htstep/htstep_split_omni.json", encoding="utf-8"))
for sp in ("train", "val"):
    c, it = S["counts"][sp], S["items"][sp]
    print(f"{sp}: json {c['videos_in_json']} video / {c['steps_in_json']} bước; sau luật bước {c['steps_after_step_filter']} bước"
          f" | bỏ YC2 {c.get('videos_yc2_overlap', 0)} video ({c.get('steps_yc2_overlap', 0)} bước), không có mp4 "
          f"{c.get('videos_no_mp4', 0)} video ({c.get('steps_no_mp4', 0)} bước) | giữ {c['videos_kept']} video / "
          f"{c['steps_kept']} bước | Full Match {it['full_match_pct']} % | text khác nhau {it['distinct_text']} | "
          f"cặp trùng text / batch 32: {it['same_text_pairs_per_batch32']}")
print("video bị loại theo lý do:", dict(Counter(S["excluded"].values())))
shutil.copy(f"{REPO}/data/htstep/htstep_split_omni.json", DRIVE_LOG)

In [ ]:
# ===== 3. Kiểm chuyển mã trên 5 video AV1: khung decord đọc từ bản H.264 == khung ffmpeg giải thẳng từ AV1 (từng pixel)
rc = sh(f"python scripts/precut_htstep.py check-transcode --n 5 --work {WORK}/check_transcode",
        log=f"{DRIVE_LOG}/check_transcode.log")
if os.path.exists(f"{WORK}/check_transcode/check_transcode.json"):
    shutil.copy(f"{WORK}/check_transcode/check_transcode.json", DRIVE_LOG)
assert rc == 0, "CHECK transcode_pixels FAIL: dừng, gửi check_transcode.json + check_transcode.log cho agent"

In [ ]:
# ===== 4. Cắt clip, tự resume từ trạng thái trên HF. Phiên bị ngắt: mở lại notebook, chạy ô 0, 1 rồi ô này
# In mỗi video xong: số clip, shard đang gom, thời gian đã chạy, ước lượng còn lại (theo số bước).
if not os.path.exists(f"{WAVE_HOME}/WAVE-7B/preprocessor_config.json"):
    sh(f'hf download nguyenminh04/omni-model --include "WAVE-7B/*.json" --local-dir {WAVE_HOME}')
rc = sh(f"python scripts/precut_htstep.py run --hf-repo {HF_CLIPS} --processor {WAVE_HOME}/WAVE-7B --work {WORK} "
        f"--jobs {JOBS}", log=f"{DRIVE_LOG}/precut_{time.strftime('%Y%m%d_%H%M')}.log")
print("exit", rc, "| xong khi dòng đầu báo '0 to cut'; còn video fetch_error thì chạy lại ô này một lần")

In [ ]:
# ===== 5. Kiểm toàn vẹn (verify) + manifest train / val (finalize), upload lên repo HF clip (meta/)
rc = sh(f"python scripts/precut_htstep.py verify --hf-repo {HF_CLIPS} --out {WORK}/precut_index.jsonl",
        log=f"{DRIVE_LOG}/verify.log")
assert rc == 0, "CHECK precut_integrity FAIL: xem verify.log (còn bước 'missing' thì chạy lại ô 4)"
rc = sh(f"python scripts/convert_htstep.py finalize --precut-index {WORK}/precut_index.jsonl --out-dir data/htstep",
        log=f"{DRIVE_LOG}/finalize.log")
assert rc == 0, "finalize lỗi: gửi finalize.log cho agent"
from huggingface_hub import HfApi, CommitOperationAdd
D = f"{REPO}/data/htstep"
files = [f"{D}/{n}" for n in ("train_omni_htstep.jsonl", "val_omni_htstep.jsonl", "htstep_split_omni.json",
                              "items_train.jsonl", "items_val.jsonl", "dropped_steps.json")]
files += [f"{WORK}/precut_index.jsonl", f"{WORK}/precut_index_summary.json"]
files += [p for p in [f"{WORK}/check_transcode/check_transcode.json"] if os.path.exists(p)]
HfApi().create_commit(HF_CLIPS, repo_type="dataset", commit_message="meta: manifests + split (convert_htstep finalize)",
                      operations=[CommitOperationAdd(f"meta/{os.path.basename(p)}", p) for p in files])
os.makedirs(f"{DRIVE_LOG}/meta", exist_ok=True)
for p in files:
    shutil.copy(p, f"{DRIVE_LOG}/meta")
    print(f"meta/{os.path.basename(p):28s} sha256 {sha256(p)}")
st = json.load(open(f"{D}/dropped_steps.json", encoding="utf-8"))
print("video bị loại lúc cắt:", st["videos_by_reason"])
for sp, s in st["stats"].items():
    print(f"{sp}: {s['steps']} bước / {s['videos']} video | trạng thái {s['status']} | kẹp end {s['clamped_end']} | "
          f"Full Match {s['full_match_pct']} % | độ dài (s) {s['length_sec']} | cặp trùng text / batch 32 "
          f"{s['same_text_pairs_per_batch32']}")
print("-> gửi cho agent:", DRIVE_LOG, "(verify.log, finalize.log, meta/)")